In [ ]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")

# Lead Generation Crew

## What We're Building

An outbound lead generation pipeline with four specialized agents:

| Agent | Role |
|-------|------|
| ICP Finder | Define the Ideal Customer Profile from product description |
| Company Researcher | Find and profile target companies matching the ICP |
| Outreach Drafter | Draft role-aware outreach angles and first-touch structure |
| Personalization Agent | Convert outreach drafts into personalized emails per company/contact |

## Crew Task Handoff

The crew runs in sequence, and each task hands off its output to the next stage:

1. ICP task produces targeting criteria.
2. Company task uses the ICP to build target account profiles.
3. Outreach drafting task uses company profiles to prepare message angles.
4. Personalization task uses both company and outreach outputs via task context.

This demonstrates both sequential handoff and explicit dependency handoff.

## Stack
- CrewAI - multi-agent pipeline
- Ollama - local LLM

In [ ]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")

## Step 2 — Product Description

In [ ]:
product_info = """
Product: DataSync Pro
Category: B2B SaaS — Data Integration Platform
What it does: Connects disparate data sources (Salesforce, HubSpot, Snowflake,
PostgreSQL, Google Sheets) and syncs them in real-time with no-code connectors.
Key differentiator: Handles schema drift automatically; 10-minute setup vs.
competitors' multi-day onboarding.
Pricing: $499/mo (Starter), $1,499/mo (Growth), Custom (Enterprise)
Best for: Mid-market companies (50-500 employees) with messy data spread
across 5+ tools who are tired of broken integrations and manual CSV exports.
"""

print("Product loaded: DataSync Pro")

## Step 3 — Create Agents

In [ ]:
if CREWAI_AVAILABLE:
    icp_finder = Agent(
        role="Ideal Customer Profile Strategist",
        goal="Define a precise ICP based on the product's value proposition",
        backstory=(
            "You are a growth marketing strategist who has built ICP frameworks for "
            "50+ B2B SaaS companies. You think in terms of firmographics, "
            "technographics, and psychographics."
        ),
        llm=llm,
        verbose=True,
    )

    company_researcher = Agent(
        role="Target Account Researcher",
        goal="Identify and profile companies that match the ICP",
        backstory=(
            "You are an SDR team lead who has researched thousands of accounts. "
            "You map each account's context, urgency signals, and likely pain points."
        ),
        llm=llm,
        verbose=True,
    )

    outreach_drafter = Agent(
        role="Outreach Drafter",
        goal="Draft outreach structure and value-angle drafts for each target company",
        backstory=(
            "You are a pipeline-building SDR who writes concise first-touch drafts "
            "aligned to role priorities and business outcomes."
        ),
        llm=llm,
        verbose=True,
    )

    personalization_agent = Agent(
        role="Personalization Agent",
        goal="Turn outreach drafts into highly personalized, reply-focused emails",
        backstory=(
            "You are a top-performing outbound specialist who personalizes each message "
            "with company-specific and role-specific context while keeping copy short."
        ),
        llm=llm,
        verbose=True,
    )
else:
    icp_finder = {"role": "ICP Finder"}
    company_researcher = {"role": "Company Researcher"}
    outreach_drafter = {"role": "Outreach Drafter"}
    personalization_agent = {"role": "Personalization Agent"}

print("4 agents created: ICP Finder, Company Researcher, Outreach Drafter, Personalization Agent")

## Step 4 — Create Tasks with Dependencies

In [ ]:
if CREWAI_AVAILABLE:
    icp_task = Task(
        description=f"""Define the Ideal Customer Profile for this product:

{product_info}

Create a detailed ICP including:
1. Firmographics: industry, company size, revenue range, geography
2. Technographics: tools they likely use, tech maturity level
3. Pain points: specific problems they face that our product solves
4. Buying triggers: events that would make them search for a solution
5. Anti-ICP: who should we NOT target (and why)""",
        expected_output="A detailed Ideal Customer Profile document.",
        agent=icp_finder,
    )

    company_task = Task(
        description="""Based on the ICP defined above, generate 5 fictional but realistic
    target company profiles. For each company include:
    1. Company name, industry, headquarters, employee count, revenue estimate
    2. Their likely tech stack (CRM, data warehouse, analytics tools)
    3. Recent news or events (funding, hiring, product launch) that create urgency
    4. Specific pain points relevant to our product
    5. Score each company 1-10 on ICP fit""",
        expected_output="5 detailed target company profiles with ICP fit scores.",
        agent=company_researcher,
    )

    outreach_task = Task(
        description="""Using the company profiles, draft outreach angles for each company:
    1. Best initial outreach angle for the likely decision-maker
    2. One concise first-touch message skeleton
    3. One follow-up skeleton with a low-friction CTA
    4. Objection to preempt for each company
    5. Business outcome framing to emphasize""",
        expected_output="Outreach draft set for each target company.",
        agent=outreach_drafter,
    )

    personalization_task = Task(
        description="""Write personalized cold outreach emails for each target company.

For each company, write:
1. Subject line (under 50 chars, personalized, no spam words)
2. Email body (under 150 words, 3 paragraphs max)
3. Follow-up email (shorter than first touch)

Rules: No buzzwords, no generic opener, no attachments.""",
        expected_output="Personalized cold emails and follow-ups for each company.",
        agent=personalization_agent,
        context=[company_task, outreach_task],
    )
else:
    icp_task = {"name": "ICP task"}
    company_task = {"name": "Company research task"}
    outreach_task = {"name": "Outreach drafting task"}
    personalization_task = {"name": "Personalization task"}

print("4 tasks created with handoff: icp -> company -> outreach -> personalization")
print("Personalization task uses explicit context from company and outreach tasks.")

## Step 5 — Run the Crew

In [ ]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Personalized outreach package ready for 5 target companies."
        self.tasks_output = [
            _DemoTaskOutput("ICP", "ICP: mid-market SaaS and data-heavy ops teams with integration pain."),
            _DemoTaskOutput("Company Research", "5 company profiles with urgency triggers and fit scores."),
            _DemoTaskOutput("Outreach Drafts", "Company-specific first-touch and follow-up message structures."),
            _DemoTaskOutput("Personalized Emails", "Final personalized subject lines and email bodies per target."),
        ]


if CREWAI_AVAILABLE:
    lead_crew = Crew(
        agents=[icp_finder, company_researcher, outreach_drafter, personalization_agent],
        tasks=[icp_task, company_task, outreach_task, personalization_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Lead generation crew assembled. Starting pipeline...")
    result = lead_crew.kickoff()
    print("\nLead generation complete.")
else:
    print("Demo mode: showing crew task handoff without live CrewAI kickoff.")
    result = _DemoResult()

In [ ]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("PERSONALIZED OUTREACH OUTPUT")
print("=" * 60)
print(result.raw)

preview("ICP Output", result.tasks_output[0].raw)
preview("Company Research Output", result.tasks_output[1].raw)
preview("Outreach Drafting Output", result.tasks_output[2].raw)
preview("Personalization Output", result.tasks_output[3].raw)

## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Crew handoff | Sequential tasks: ICP -> Company Research -> Outreach Drafting -> Personalization |
| Explicit dependency handoff | `context=[company_task, outreach_task]` for personalization task |
| Role specialization | Each agent has a focused scope and output format |
| Personalization quality | Final emails combine account context plus outreach draft strategy |

## Extensions

- CRM integration: push generated leads and drafts to HubSpot/Salesforce
- Validation gate: add a reviewer task for tone and compliance checks
- A/B variants: generate two personalized email variants per target
- Prioritization: add a lead scoring task before personalization